In [1]:
import argparse
import json
from pathlib import Path
import os
import time
from datetime import timedelta
from typing import Dict, List, Optional, Sequence, Tuple
import sys
sys.path.append("../..")

import numpy as np
import csv
import torch
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F
from monai import transforms
from monai.data import CacheDataset, DataLoader, ThreadDataLoader
from monai.data.utils import pad_list_data_collate
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import random

from monai.inferers import DiffusionInferer
from monai.networks.nets import DiffusionModelUNet
from monai.networks.schedulers import DDPMScheduler

from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

import matplotlib.pyplot as plt

import utils.custom_transforms as custom_transforms
from utils.utils import *
import AnoDDPM.simplex as simplex
import utils.simplex_ddpm as simplex_ddpm

In [2]:
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"

In [3]:
def compute_loss_simplex(images, simplexObj, model, inferer, num_timesteps, device, return_pred=False):
    with autocast("cuda", enabled=True):
        # Generate random noise
 
        noise = simplex_ddpm.generate_simplex_noise(simplexObj, images.shape, normalize=False).to(device, non_blocking=True) #TODO: check non_blocking (21/09/2025)
 
        # Create timesteps
        timesteps = torch.randint(0, num_timesteps, (images.shape[0],), device=images.device).long()

        # Get model prediction
        noise_pred = inferer(inputs=images, diffusion_model=model, noise=noise, timesteps=timesteps)

        loss = F.mse_loss(noise_pred.float(), noise.float())
        if return_pred:
            return loss, noise_pred, noise
        return loss


def compute_loss_gaussian(images, model, inferer, num_timesteps, device, return_pred=False):
    with autocast("cuda", enabled=True):
        # Generate random noise
        noise = torch.randn_like(images).to(device, non_blocking=True) #TODO: check non_blocking (21/09/2025)

        # Create timesteps
        timesteps = torch.randint(0, num_timesteps, (images.shape[0],), device=images.device).long()

        # Get model prediction
        noise_pred = inferer(inputs=images, diffusion_model=model, noise=noise, timesteps=timesteps)

        loss = F.mse_loss(noise_pred.float(), noise.float())
        if return_pred:
            return loss, noise_pred, noise
        return loss


In [4]:

def _diffusion_step(images, noise_type, simplexObj, model, inferer, num_timesteps, device, return_pred):
    if noise_type == "simplex":
        return compute_loss_simplex(images, simplexObj, model, inferer, num_timesteps, device, return_pred)
    return compute_loss_gaussian(images, model, inferer, num_timesteps, device, return_pred)


In [5]:
def _generate_patch_slices(spatial_shape: Sequence[int], patch_size: Sequence[int], overlap: Sequence[int]):


    ranges: List[List[int]] = []

    for dim, size, ov in zip(spatial_shape, patch_size, overlap):
        step = max(size - ov, 1)

        if dim <= size:
            coords = [0]
        else:
            coords = list(range(0, max(dim - size, 0) + 1, step))
            if coords[-1] != dim - size:
                coords.append(dim - size)
        ranges.append(coords)
        
    for h in ranges[0]:
        for w in ranges[1]:
            for d in ranges[2]:
                yield (slice(h, h + patch_size[0]), slice(w, w + patch_size[1]), slice(d, d + patch_size[2]))



In [6]:
@torch.no_grad()
def my_sample(model, device, noise_type, simplexObj, image, infer_scheduler, timesteps, return_intermediates=False):

    if noise_type == "simplex":
        noise = simplex_ddpm.generate_simplex_noise(simplexObj, image.shape, normalize=False).to(device)
    if noise_type == "gaussian":
        noise = torch.randn(image.shape).to(device)


    timesteps_list = torch.Tensor([timesteps for a in range(image.shape[0])]).to(image.device).long()

    image = infer_scheduler.add_noise(image, noise, timesteps_list).to(device) #TODO


    intermediates = []
    intermediates_step = 20

            
    for t in tqdm(range(timesteps, 0, -1)): # va de timesteps à 0
        
        model_output = model(
            image, timesteps=torch.Tensor((t,)).to(device), context=None
        )
        #print(model_output.shape)
        
        image, _ = infer_scheduler.step(model_output, t, image)
    
        if (t== timesteps-1 or t%intermediates_step == 0) and return_intermediates:
            intermediates.append(image)

    if return_intermediates:
        return image, intermediates
    else:
        return image

In [ ]:

def _create_patch_weight(patch_size: Sequence[int], sigma_scale: float = 0.125) -> torch.Tensor:
    """
    Create a 3D Gaussian weight map that gives more importance to the center of the patch.
    This helps blend overlapping patches smoothly and eliminates seam artifacts.
    """
    weight = torch.ones(patch_size)
    
    for dim in range(3):
        size = patch_size[dim]

        # Create 1D Gaussian-like weight using cosine tapering
        # This gives weight 1 at center and smoothly decreases to ~0.5 at edges
        coords = torch.linspace(0, 1, size)

        # Cosine window (Hann-like): smooth transition from edges to center
        window = 0.5 * (1 - torch.cos(2 * np.pi * coords))

        #window = window * 0.5 + 0.5  # Range: 0.5 to 1.0
        window = window * 0.9 + 0.1  # Range: 0.9 to 1.0

        # Reshape for broadcasting
        shape = [1, 1, 1]
        shape[dim] = size
        window = window.view(shape)
        
        weight = weight * window
    
    return weight


def _run_patchwise_test(
    volume: torch.Tensor,
    patch_size: Sequence[int],
    overlap: Sequence[int],
    patch_batch_size: int,
    noise_type: str,
    simplexObj,
    model,
    infer_scheduler,
    num_timesteps: int,
    device,
):
    aggregator_pred = torch.zeros_like(volume, dtype=torch.float32)
    weight_sum = torch.zeros_like(volume, dtype=torch.float32)
    patch_queue: List[torch.Tensor] = []
    slice_queue: List[Tuple[slice, slice, slice]] = []
    total_patches = 0
    
    # Create weight map for smooth blending
    patch_weight = _create_patch_weight(patch_size).to(device)
    """
    Same as _run_patchwise_inference but goes all the way to the fully denoised image
    Uses weighted averaging to eliminate seam artifacts at patch boundaries.
    """

    def _flush_queue():
        nonlocal total_patches

        if not patch_queue:
            return
        
        batch_tensor = torch.cat(patch_queue, dim=0) # transforms all the patches into a single batch tensor

        preds = my_sample(model, device, noise_type, simplexObj, batch_tensor, infer_scheduler, num_timesteps)

        patch_count = batch_tensor.shape[0]
        total_patches += patch_count

        for idx, patch_slices in enumerate(slice_queue):
            target_slice = (slice(None), slice(None), patch_slices[0], patch_slices[1], patch_slices[2])
            
            # Apply weighted contribution instead of simple addition
            aggregator_pred[target_slice] += preds[idx].unsqueeze(0).float() * patch_weight
            weight_sum[target_slice] += patch_weight  # Accumulate weights instead of counts


        patch_queue.clear()
        slice_queue.clear()

    for patch_slices in _generate_patch_slices(volume.shape[-3:], patch_size, overlap): # goes through the slices that define each patch

        patch = volume[(slice(None), slice(None), patch_slices[0], patch_slices[1], patch_slices[2])] # extracts the patch using the slices

        patch_queue.append(patch) # patch_queue stores all the patches for the current volume batch
        slice_queue.append(patch_slices)

        if len(patch_queue) >= patch_batch_size: # makes sure there aren't too many patches at one time (memory issues)
            _flush_queue() # does the inference and computes loss

    _flush_queue()

    weight_sum = torch.clamp(weight_sum, min=1e-8)
    stitched_pred = aggregator_pred / weight_sum  # Weighted average for smooth blending

    return stitched_pred


In [22]:
config_dict = json.load(open(ROOT_DIR+"AnoDiffExperiments/experiment_2/exp_2_4/config.json", "r"))
args = argparse.Namespace(**config_dict)


In [9]:
ROOT_DIR = args.root_dir
EXPERIMENT_NAME = args.experiment_name
SUB_EXPERIMENT_NAME = args.sub_experiment_name
MODELS_DIR = ROOT_DIR+f"AnoDiffExperiments/{EXPERIMENT_NAME}/{SUB_EXPERIMENT_NAME}/models/"
os.makedirs(MODELS_DIR, exist_ok=True)


train_patch_size = args.patch_size
infer_patch_size = args.patch_size
patch_overlap = args.dataset["patch_overlap"]

patch_infer_batch_size = args.dataset["batch_size"]

ddp_bool = False

rank = 0
world_size = 1
device = 0

torch.cuda.set_device(device)
tprint(f"Using {device}")

torch.backends.cudnn.benchmark = True
torch.set_num_threads(torch.get_num_threads())
torch.autograd.set_detect_anomaly(False)


test_reconstruction_csv = os.path.join(ROOT_DIR, f"AnoDiffExperiments/data_splits_lists/{args.dataset['name']}/test.csv")
test_reconstruction_images_path = []

with open(test_reconstruction_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):
        test_reconstruction_images_path.append(ROOT_DIR+line[0])

test_reconstruction_datalist = test_reconstruction_images_path


[12:23:32] Using 0


243it [00:00, 11681.56it/s]
243it [00:00, 11681.56it/s]


In [10]:

batch_size = args.dataset["batch_size"]
num_workers = args.dataset["num_workers"]

test_reconstruction_transforms = define_instance(args, "val_transforms")
test_reconstruction_ds = CacheDataset(data=test_reconstruction_datalist[:batch_size], transform=test_reconstruction_transforms) #TODO: val_datalist[:batch_size]

ddp_bool = False

if ddp_bool:
    test_sampler = torch.utils.data.distributed.DistributedSampler(test_reconstruction_ds, num_replicas=world_size, rank=rank)
else:
    test_sampler = None

test_reconstruction_loader = DataLoader( # smaller batch size for validation since we are validating on full volumes
    test_reconstruction_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True, sampler=test_sampler
)

Loading dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:42<00:00,  1.51it/s]


In [11]:
model = define_instance(args, "network_def").to(device)
model.load_state_dict(torch.load(MODELS_DIR+f"{SUB_EXPERIMENT_NAME}_best_model.pth", map_location="cuda:0"))
model.eval()

DiffusionModelUNet(
  (conv_in): Convolution(
    (conv): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  )
  (time_embed): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (down_blocks): ModuleList(
    (0): DownBlock(
      (resnets): ModuleList(
        (0-1): 2 x DiffusionUNetResnetBlock(
          (norm1): GroupNorm(32, 32, eps=1e-06, affine=True)
          (nonlinearity): SiLU()
          (conv1): Convolution(
            (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          )
          (time_emb_proj): Linear(in_features=128, out_features=32, bias=True)
          (norm2): GroupNorm(32, 32, eps=1e-06, affine=True)
          (conv2): Convolution(
            (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          )
          (skip_connection): Identity()
        )
      )


In [23]:


simplexObj = None
if args.noise["type"] == "simplex":
    simplexObj = simplex.Simplex_CLASS()
    scheduler = simplex_ddpm.SimplexDDPMScheduler(num_train_timesteps=args.noise["num_timesteps_full_noise"], schedule=args.noise["schedule"], octaves=args.noise["simplex_octaves"], persistence=args.noise["simplex_persistence"], frequency=args.noise["simplex_frequency"], normalize=args.noise["normalize"])

elif args.noise["type"] == "gaussian":
    scheduler = DDPMScheduler(num_train_timesteps=args.noise["num_timesteps_full_noise"], schedule=args.noise["schedule"])

num_diffusion_steps = int(args.noise["noise_rate_train_and_infer"] * args.noise["num_timesteps_full_noise"])

if args.diffusion_train["optimizer"]["type"] == "Adam":
    optimizer = torch.optim.Adam(params=model.parameters(), lr=args.diffusion_train["optimizer"]["lr"] * world_size)

if args.diffusion_train["lr_scheduler"]!= "none":
    
    if args.diffusion_train["lr_scheduler"] == "MultiStepLR":
        lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=args.diffusion_train["lr_scheduler_milestones"],
        gamma=0.1)



inferer = DiffusionInferer(scheduler)


In [24]:
if ddp_bool:
    # When using DDP, BatchNorm needs to be converted to SyncBatchNorm.
    #model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)
    model = DDP(model, device_ids=[device], output_device=rank, find_unused_parameters=False)

if rank==0:
    os.makedirs(ROOT_DIR+f"AnoDiffExperiments/tensorboard/{SUB_EXPERIMENT_NAME}", exist_ok=True)
    writer = SummaryWriter(ROOT_DIR+f"AnoDiffExperiments/tensorboard/{SUB_EXPERIMENT_NAME}")

max_epochs = args.diffusion_train["max_epochs"]
val_interval = args.diffusion_train["val_interval"]

best_val_epoch_loss = np.inf
best_val_epoch = 0

scaler = GradScaler("cuda")

In [25]:
test_diffusion_steps = 200

with torch.no_grad():
    for step, batch in enumerate(test_reconstruction_loader):
        if step>0:break
        
        images = batch.to(device)
        images = images[..., args.slice_indexes_start:args.slice_indexes_end]
        
        volumes = images.shape[0]

        for idx in range(volumes):
            if idx>0:break

            volume = images[idx : idx + 1]
            volume = volume.to(device)
            
            stitched_pred = _run_patchwise_test(
                volume,
                infer_patch_size,
                patch_overlap,
                patch_infer_batch_size,
                args.noise["type"],
                simplexObj,
                model,
                scheduler,
                test_diffusion_steps,
                device,
            )

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [16:12<00:00,  4.86s/it]



In [26]:
stitched_pred_normalized = torch.clamp(scale_intensity_from_histogram_peak(stitched_pred, 2.0/7.0), 0.0, 1.0)

In [27]:
stitched_pred_normalized.shape

torch.Size([1, 1, 128, 128, 72])

In [28]:

# Get the first volume and its stitched prediction
vol = volume[0, 0].cpu().numpy()  # Shape: (H, W, D)
pred = stitched_pred_normalized[0, 0].cpu().numpy()  # Shape: (H, W, D)


faire le post processing où je recale le pic d'histogramme

In [17]:

# Get middle slice indices
mid_axial = vol.shape[2] // 2
mid_coronal = vol.shape[1] // 2
mid_sagittal = vol.shape[0] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Original volume
axes[0, 0].imshow(vol[:, :, mid_axial], cmap='gray')
axes[0, 0].set_title('Original - Axial')
axes[0, 0].axis('off')

axes[0, 1].imshow(vol[:, mid_coronal, :], cmap='gray')
axes[0, 1].set_title('Original - Coronal')
axes[0, 1].axis('off')

axes[0, 2].imshow(vol[mid_sagittal, :, :], cmap='gray')
axes[0, 2].set_title('Original - Sagittal')
axes[0, 2].axis('off')

# Stitched prediction
axes[1, 0].imshow(pred[:, :, mid_axial], cmap='gray')
axes[1, 0].set_title('Stitched Pred - Axial')
axes[1, 0].axis('off')

axes[1, 1].imshow(pred[:, mid_coronal, :], cmap='gray')
axes[1, 1].set_title('Stitched Pred - Coronal')
axes[1, 1].axis('off')

axes[1, 2].imshow(pred[mid_sagittal, :, :], cmap='gray')
axes[1, 2].set_title('Stitched Pred - Sagittal')
axes[1, 2].axis('off')

plt.suptitle(f"{test_diffusion_steps} timesteps")
plt.tight_layout()
plt.show()

In [ ]:

# Get middle slice indices
mid_axial = vol.shape[2] // 2
mid_coronal = vol.shape[1] // 2
mid_sagittal = vol.shape[0] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Original volume
axes[0, 0].imshow(vol[:, :, mid_axial], cmap='gray')
axes[0, 0].set_title('Original - Axial')
axes[0, 0].axis('off')

axes[0, 1].imshow(vol[:, mid_coronal, :], cmap='gray')
axes[0, 1].set_title('Original - Coronal')
axes[0, 1].axis('off')

axes[0, 2].imshow(vol[mid_sagittal, :, :], cmap='gray')
axes[0, 2].set_title('Original - Sagittal')
axes[0, 2].axis('off')

# Stitched prediction
axes[1, 0].imshow(pred[:, :, mid_axial], cmap='gray')
axes[1, 0].set_title('Stitched Pred - Axial')
axes[1, 0].axis('off')

axes[1, 1].imshow(pred[:, mid_coronal, :], cmap='gray')
axes[1, 1].set_title('Stitched Pred - Coronal')
axes[1, 1].axis('off')

axes[1, 2].imshow(pred[mid_sagittal, :, :], cmap='gray')
axes[1, 2].set_title('Stitched Pred - Sagittal')
axes[1, 2].axis('off')

plt.suptitle(f"weight map with cosine tapering (window from 0.5 to 1.0) and {test_diffusion_steps} timesteps")
plt.tight_layout()
plt.show()

In [29]:

# Get middle slice indices
mid_axial = vol.shape[2] // 2
mid_coronal = vol.shape[1] // 2
mid_sagittal = vol.shape[0] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Original volume
axes[0, 0].imshow(vol[:, :, mid_axial], cmap='gray')
axes[0, 0].set_title('Original - Axial')
axes[0, 0].axis('off')

axes[0, 1].imshow(vol[:, mid_coronal, :], cmap='gray')
axes[0, 1].set_title('Original - Coronal')
axes[0, 1].axis('off')

axes[0, 2].imshow(vol[mid_sagittal, :, :], cmap='gray')
axes[0, 2].set_title('Original - Sagittal')
axes[0, 2].axis('off')

# Stitched prediction
axes[1, 0].imshow(pred[:, :, mid_axial], cmap='gray')
axes[1, 0].set_title('Stitched Pred - Axial')
axes[1, 0].axis('off')

axes[1, 1].imshow(pred[:, mid_coronal, :], cmap='gray')
axes[1, 1].set_title('Stitched Pred - Coronal')
axes[1, 1].axis('off')

axes[1, 2].imshow(pred[mid_sagittal, :, :], cmap='gray')
axes[1, 2].set_title('Stitched Pred - Sagittal')
axes[1, 2].axis('off')

plt.suptitle(f"weight map with cosine tapering (window from 0.1 to 1.0) and {test_diffusion_steps} timesteps")
plt.tight_layout()
plt.show()

In [30]:
patch_weight = _create_patch_weight((64, 64, 64)).to(device)

In [35]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

pw = patch_weight.cpu().numpy()
mid = pw.shape[0] // 2

axes[0].imshow(pw[:, :, mid], cmap='grey')
axes[0].set_title('Axial (z-mid)')
#axes[0].axis('off')

axes[1].imshow(pw[:, mid, :], cmap='grey')
axes[1].set_title('Coronal (y-mid)')
#axes[1].axis('off')

axes[2].imshow(pw[mid, :, :], cmap='grey')
axes[2].set_title('Sagittal (x-mid)')
#axes[2].axis('off')

plt.suptitle('Patch Weight Map')
plt.colorbar(axes[0].images[0], ax=axes, orientation='vertical', fraction=0.05)
#plt.tight_layout()
plt.show()